In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/results.csv
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/2715746315.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/3463034205.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/268704620.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/2673564214.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/7535037918.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/4912369161.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/4828071602.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/6802728196.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/3346289227.jpg
/kaggle/input/datasets/raghudinkavijaykum

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("raghudinkavijaykumar/flickr-images-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset


In [4]:
import os

base_path = "/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset"

print(os.listdir(base_path))

['flickr30k_images', 'results.csv']


In [5]:
!len(/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset)

/bin/bash: -c: line 1: syntax error near unexpected token `/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset'
/bin/bash: -c: line 1: `len(/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset)'


In [6]:
    print(os.listdir(base_path + "/flickr30k_images"))


['2715746315.jpg', '3463034205.jpg', '268704620.jpg', '2673564214.jpg', '7535037918.jpg', '4912369161.jpg', '4828071602.jpg', '6802728196.jpg', '3346289227.jpg', '3217056901.jpg', '272471327.jpg', '4717261252.jpg', '4763916790.jpg', '2700788458.jpg', '2795287622.jpg', '4453893059.jpg', '2094323311.jpg', '2375770917.jpg', '5962278982.jpg', '2460568004.jpg', '2567962271.jpg', '4569787426.jpg', '2782433864.jpg', '4571040008.jpg', '7616312438.jpg', '4905053758.jpg', '3533922605.jpg', '4532518749.jpg', '3397633339.jpg', '4544052868.jpg', '3173976185.jpg', '2148680620.jpg', '2836325261.jpg', '4636055948.jpg', '2744615692.jpg', '5345473274.jpg', '3551637285.jpg', '4463211241.jpg', '8127485677.jpg', '3135826945.jpg', '393959556.jpg', '165984091.jpg', '3284899112.jpg', '4946045875.jpg', '4535251327.jpg', '4355004642.jpg', '4921374490.jpg', '188244881.jpg', '4860415782.jpg', '3497565955.jpg', '2267682214.jpg', '6763370245.jpg', '2346564235.jpg', '3802499653.jpg', '7001949951.jpg', '2203586218.jp

In [8]:
dataset_images_path = "/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images"

captions_file = "/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/captions.txt"

In [ ]:
"""
Image Captioning using InceptionV3 + LSTM (Kaggle Version)
"""

import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gc
import re
import pickle
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import random

# ===============================
# Random Image
# ===============================

def get_random_image(folder_path):
    images = [img for img in os.listdir(folder_path)
              if img.lower().endswith((".jpg", ".jpeg", ".png"))]

    if len(images) == 0:
        raise ValueError("No images found")

    return os.path.join(folder_path, random.choice(images))


# ===============================
# Paths (KAGGLE)
# ===============================

dataset_images_path = "/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/flickr30k_images"
captions_file = "/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/results.csv"

features_cache = "/kaggle/working/image_features.pkl"
encoder_model_file = "/kaggle/working/encoder_model.weights.h5"
decoder_model_file = "/kaggle/working/decoder_model.weights.h5"

# ===============================
# Config
# ===============================

img_height = 224
img_width = 224

BATCH_SIZE = 32
EPOCHS = 5   # keep small for demo

EMBEDDING_DIM = 256
UNITS = 256
TOP_K = 2000
FEATURE_DIM = 2048
validation_split = 0.2

# ===============================
# Caption preprocessing
# ===============================

def preprocess_caption(caption):
    caption = caption.lower()
    caption = re.sub(r"[^a-z ]", "", caption)
    caption = re.sub(r"\s+", " ", caption).strip()

    if len(caption.split()) < 3:
        return None

    return "<start> " + caption + " <end>"

# ===============================
# Load captions (CSV)
# ===============================

# print("Loading captions...")

# df = pd.read_csv(captions_file)

# images_captions_dict = {}

# for _, row in df.iterrows():
#     image_name = row['image_name']
#     caption = preprocess_caption(row['comment'])

#     if caption is None:
#         continue

#     images_captions_dict.setdefault(image_name, []).append(caption)

images_captions_dict = {}

with open(captions_file, "r", encoding="utf-8") as f:

    next(f)  # skip header

    for line in f:

        parts = line.strip().split("|")

        if len(parts) < 3:
            continue

        image_name = parts[0].strip()   # correct image name
        caption_text = parts[2].strip() # actual caption

        caption = preprocess_caption(caption_text)

        if caption is None:
            continue

        images_captions_dict.setdefault(image_name, []).append(caption)
# images_captions_dict = {}

# with open(captions_file, "r", encoding="utf-8") as f:
    
#     next(f)  # skip header if present

#     for line in f:
#         parts = line.strip().split(",", 1)

#         if len(parts) < 2:
#             continue

#         image_id = parts[0].split("#")[0]   # remove #0, #1
#         caption = preprocess_caption(parts[1])

#         if caption is None:
#             continue

#         images_captions_dict.setdefault(image_id, []).append(caption)

# print("Total images:", len(images_captions_dict))

# ===============================
# Feature extractor
# ===============================

print("Checking sample path:")
print(os.path.join(dataset_images_path, list(images_captions_dict.keys())[0]))

def build_feature_extractor():
    base = tf.keras.applications.InceptionV3(
        include_top=False,
        weights="imagenet",
        input_shape=(img_height, img_width, 3),
    )

    out = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    return tf.keras.Model(base.input, out)

# ===============================
# Extract OR Load Features
# ===============================

if os.path.exists(features_cache):
    print("Loading cached features...")
    with open(features_cache, "rb") as f:
        images_features = pickle.load(f)

else:
    print("Extracting features...")
    extractor = build_feature_extractor()

    images_features = {}

    for img_name in tqdm(images_captions_dict):
        path = os.path.join(dataset_images_path, img_name)

        try:
            raw = tf.io.read_file(path)
            img = tf.image.decode_jpeg(raw, channels=3)
            img = tf.image.resize(img, (img_height, img_width))
            img = tf.keras.applications.inception_v3.preprocess_input(img)
            img = tf.expand_dims(img, 0)

            feat = extractor(img)
            images_features[img_name] = feat.numpy()[0]

        except:
            continue

    with open(features_cache, "wb") as f:
        pickle.dump(images_features, f)

    del extractor
    gc.collect()

print("Features ready:", len(images_features))

# ===============================
# Dataset
# ===============================

image_filenames = list(images_features.keys())

train_imgs, test_imgs = train_test_split(
    image_filenames,
    test_size=validation_split,
    random_state=42
)

# 🔥 Reduce for faster training
train_imgs = train_imgs[:5000]

def build_pairs(img_list):
    X, Y = [], []

    for img in img_list:
        feat = images_features[img]
        for cap in images_captions_dict[img]:
            X.append(feat)
            Y.append(cap)

    return X, Y

X_train, y_train_raw = build_pairs(train_imgs)

# ===============================
# Tokenizer
# ===============================

tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=TOP_K,
    oov_token="<unk>"
)

tokenizer.fit_on_texts(y_train_raw)

tokenizer.word_index["<pad>"] = 0
tokenizer.index_word[0] = "<pad>"

y_train_seq = tokenizer.texts_to_sequences(y_train_raw)

y_train_pad = tf.keras.preprocessing.sequence.pad_sequences(
    y_train_seq,
    padding="post"
)

vocab_size = min(TOP_K, len(tokenizer.word_index) + 1)

dataset = (
    tf.data.Dataset.from_tensor_slices((np.array(X_train), y_train_pad))
    .shuffle(1024)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ===============================
# Models
# ===============================

class CNN_Encoder(tf.keras.Model):
    def __init__(self, embedding_dim):
        super().__init__()
        self.fc = tf.keras.layers.Dense(embedding_dim, activation="relu")

    def call(self, x):
        return self.fc(x)

class RNN_Decoder(tf.keras.Model):
    def __init__(self, embedding_dim, units, vocab_size):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True)
        self.fc = tf.keras.layers.Dense(vocab_size)

    def call(self, x, features):
        x = self.embedding(x)
        features = tf.expand_dims(features, 1)
        x = tf.concat([features, x], axis=1)
        output = self.lstm(x)
        return self.fc(output)

enc = CNN_Encoder(EMBEDDING_DIM)
dec = RNN_Decoder(EMBEDDING_DIM, UNITS, vocab_size)

optimizer = tf.keras.optimizers.Adam()

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ===============================
# Training
# ===============================

if os.path.exists(encoder_model_file) and os.path.exists(decoder_model_file):
    print("Loading saved model...")
    enc.load_weights(encoder_model_file)
    dec.load_weights(decoder_model_file)

else:
    print("Training model...")

    for epoch in range(EPOCHS):
        total_loss = 0

        for img_feat, target in dataset:

            with tf.GradientTape() as tape:
                features = enc(img_feat)
                predictions = dec(target[:, :-1], features)

                loss = loss_object(target[:, 1:], predictions[:, 1:, :])
                loss = tf.reduce_mean(loss)

            vars = enc.trainable_variables + dec.trainable_variables
            grads = tape.gradient(loss, vars)
            optimizer.apply_gradients(zip(grads, vars))

            total_loss += loss

        print(f"Epoch {epoch+1}, Loss: {total_loss.numpy()}")

    enc.save_weights(encoder_model_file)
    dec.save_weights(decoder_model_file)

# ===============================
# Caption Generation
# ===============================

extractor = build_feature_extractor()

def generate_caption(image_path):
    raw = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, (img_height, img_width))
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    img = tf.expand_dims(img, 0)

    feature = extractor(img)
    feature = enc(feature)

    result = []
    input_seq = [tokenizer.word_index['start']]

    for _ in range(15):
        seq = tf.expand_dims(input_seq, 0)
        pred = dec(seq, feature)
        pred_id = tf.argmax(pred[0, -1]).numpy()

        word = tokenizer.index_word.get(pred_id, "")

        if word == "end":
            break

        result.append(word)
        input_seq.append(pred_id)

    return " ".join(result)

# ===============================
# Demo
# ===============================

sample_path = get_random_image(dataset_images_path)

print("\nRandom Image:", sample_path)

caption = generate_caption(sample_path)

print("\nGenerated Caption:")
print(caption)

# ===============================
# Save tokenizer
# ===============================

with open("/kaggle/working/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("\nSaved all files to /kaggle/working")

In [ ]:
"""
Image Captioning using InceptionV3 + LSTM (Kaggle Ready - FINAL)
"""

import os
import gc
import re
import pickle
import random
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image

# ===============================
# KAGGLE PATHS (IMPORTANT)
# ===============================

BASE_DIR = "/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset"

dataset_images_path = "/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images"
captions_file = BASE_DIR + "/results.csv"

# Save outputs here
WORK_DIR = "/kaggle/working"

features_cache = WORK_DIR + "/image_features.pkl"
encoder_model_file = WORK_DIR + "/encoder_model.weights.h5"
decoder_model_file = WORK_DIR + "/decoder_model.weights.h5"

# ===============================
# FORCE DELETE OLD CACHE (IMPORTANT)
# ===============================

if os.path.exists(features_cache):
    print("Deleting old cached features...")
    os.remove(features_cache)

# ===============================
# CONFIG
# ===============================

img_height = 224
img_width = 224

BATCH_SIZE = 8
EPOCHS = 10   # reduce for speed (increase later)

EMBEDDING_DIM = 256
UNITS = 256
TOP_K = 5000
FEATURE_DIM = 2048
MAX_CAPTION_LENGTH = 20

validation_split = 0.2

# ===============================
# RANDOM IMAGE FUNCTION
# ===============================

def get_random_image(folder_path):
    images = [img for img in os.listdir(folder_path) if img.endswith(".jpg")]
    return os.path.join(folder_path, random.choice(images))

# ===============================
# CAPTION PREPROCESS
# ===============================

def preprocess_caption(caption):
    caption = caption.lower()
    caption = re.sub(r"[^a-z ]", "", caption)
    caption = re.sub(r"\s+", " ", caption).strip()
    return "<start> " + caption + " <end>"

# ===============================
# LOAD CAPTIONS (FIXED)
# ===============================

print("Loading captions...")

images_captions_dict = {}

with open(captions_file, "r", encoding="utf-8") as f:
    
    next(f)  # skip header

    for line in f:
        parts = line.strip().split("|")

        if len(parts) < 3:
            continue

        image_name = parts[0].strip()
        caption_text = parts[2].strip()

        caption = preprocess_caption(caption_text)

        images_captions_dict.setdefault(image_name, []).append(caption)

print("Total images:", len(images_captions_dict))

# ===============================
# FEATURE EXTRACTOR
# ===============================

def build_feature_extractor():
    base = tf.keras.applications.InceptionV3(
        include_top=False,
        weights="imagenet",
        input_shape=(img_height, img_width, 3),
    )
    out = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    return tf.keras.Model(base.input, out)

# ===============================
# EXTRACT FEATURES
# ===============================

print("Checking sample image path:")

for key in list(images_captions_dict.keys())[:5]:
    print(os.path.join(dataset_images_path, key))

print("Extracting features...")

extractor = build_feature_extractor()
images_features = {}

for img_name in tqdm(images_captions_dict.keys()):
    img_path = os.path.join(dataset_images_path, img_name)

    try:
        img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
        img = tf.keras.preprocessing.image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = tf.keras.applications.inception_v3.preprocess_input(img)

        feature = extractor.predict(img, verbose=0)
        images_features[img_name] = feature[0]

    except:
        continue

# SAVE FEATURES
with open(features_cache, "wb") as f:
    pickle.dump(images_features, f)

print("Features ready:", len(images_features))

# ===============================
# DATA PREP
# ===============================

image_filenames = list(images_features.keys())

train_imgs, test_imgs = train_test_split(
    image_filenames,
    test_size=validation_split,
    random_state=42
)

def build_pairs(img_list):
    X, Y = [], []

    for img in img_list:
        for cap in images_captions_dict[img]:
            X.append(images_features[img])
            Y.append(cap)

    return X, Y

X_train, y_train_raw = build_pairs(train_imgs)

# ===============================
# TOKENIZER
# ===============================

tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=TOP_K, oov_token="<unk>")
tokenizer.fit_on_texts(y_train_raw)

tokenizer.word_index["<pad>"] = 0
tokenizer.index_word[0] = "<pad>"

sequences = tokenizer.texts_to_sequences(y_train_raw)

y_train = tf.keras.preprocessing.sequence.pad_sequences(sequences, padding="post")

vocab_size = len(tokenizer.word_index) + 1

dataset = tf.data.Dataset.from_tensor_slices((np.array(X_train), y_train))
dataset = dataset.shuffle(1000).batch(BATCH_SIZE)

# ===============================
# MODEL
# ===============================

class CNN_Encoder(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.fc = tf.keras.layers.Dense(EMBEDDING_DIM)

    def call(self, x):
        return self.fc(x)

class RNN_Decoder(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, EMBEDDING_DIM)
        self.lstm = tf.keras.layers.LSTM(UNITS, return_sequences=True)
        self.fc = tf.keras.layers.Dense(vocab_size)

    def call(self, x, features):
        x = self.embedding(x)
        features = tf.expand_dims(features, 1)
        x = tf.concat([features, x], axis=1)
        x = self.lstm(x)
        return self.fc(x)

enc = CNN_Encoder()
dec = RNN_Decoder()

optimizer = tf.keras.optimizers.Adam()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ===============================
# TRAIN
# ===============================

print("Training started...")

for epoch in range(EPOCHS):
    total_loss = 0

    for img_feat, target in dataset:
        with tf.GradientTape() as tape:
            features = enc(img_feat)
            preds = dec(target[:, :-1], features)

            loss = loss_fn(target[:, 1:], preds[:, 1:, :])

        variables = enc.trainable_variables + dec.trainable_variables
        grads = tape.gradient(loss, variables)
        optimizer.apply_gradients(zip(grads, variables))

        total_loss += loss

    print(f"Epoch {epoch+1}, Loss: {total_loss.numpy()}")

# SAVE MODELS
enc.save_weights(encoder_model_file)
dec.save_weights(decoder_model_file)

print("Training Done ✅")

# ===============================
# GENERATE CAPTION
# ===============================

def generate_caption(image_path):

    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224,224))
    img = tf.keras.preprocessing.image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = tf.keras.applications.inception_v3.preprocess_input(img)

    feature = extractor.predict(img, verbose=0)
    feature = enc(feature)

    caption = "<start>"

    for i in range(MAX_CAPTION_LENGTH):
        seq = tokenizer.texts_to_sequences([caption])[0]
        seq = tf.keras.preprocessing.sequence.pad_sequences([seq], maxlen=MAX_CAPTION_LENGTH)

        pred = dec(seq, feature)
        pred_id = np.argmax(pred[0][i])

        word = tokenizer.index_word.get(pred_id, "")

        if word == "<end>":
            break

        caption += " " + word

    return caption.replace("<start>", "")

# ===============================
# DEMO
# ===============================

sample_path = get_random_image(dataset_images_path)

print("Random Image:", sample_path)


img = Image.open(sample_path)
plt.imshow(img)
plt.axis("off")

caption = generate_caption(sample_path)

plt.title(caption)
plt.show()

print("\nGenerated Caption:", caption)

2026-04-10 13:08:01.402935: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775826481.725681      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775826481.829716      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775826482.636118      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775826482.636251      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775826482.636258      55 computation_placer.cc:177] computation placer alr

Loading captions...
Total images: 31783
Checking sample image path:
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/1000092795.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/10002456.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/1000268201.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/1000344755.jpg
/kaggle/input/datasets/raghudinkavijaykumar/flickr-images-dataset/flickr30k_images/1000366164.jpg
Extracting features...


2026-04-10 13:08:36.946153: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


100%|██████████| 31783/31783 [1:37:43<00:00,  5.42it/s]


Features ready: 31783
Training started...
